### Part C answers → `outputs/ANSWERS.md`
Every number comes from a Spark SQL query on the silver / gold tables; the query is saved next to its result.

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

In [0]:
answers = ["# ANSWERS\n", "_Generated by `04_outputs/answers`. Sections marked ✍️ are written by hand after the run._\n"]


def run_query(sql):
    """Run a query and return (pandas result, markdown block with the query and its output)."""
    result = to_pandas_safe(spark.sql(sql))
    display(spark.sql(sql))
    return result, f"```sql\n{sql.strip()}\n```\n\n{md_table(result)}"


FACT, DQ, BRONZE = tbl("02_silver", "fact_trips"), tbl("02_silver", "dq_exclusions"), tbl("01_bronze", "yellow_trips_raw")
GRID, STRESS, STRESS_CONV = tbl("03_gold", "window_grid"), tbl("03_gold", "zone_stress"), tbl("03_gold", "zone_stress_conventional")

#### Reconciliation

In [0]:
recon, md = run_query(f"""
SELECT (SELECT COUNT(*) FROM {BRONZE}) AS raw_rows,
       (SELECT COUNT(*) FROM {FACT}) AS kept_rows,
       (SELECT SUM(rows_excluded) FROM {DQ}) AS excluded_rows""")
assert recon.raw_rows[0] == recon.kept_rows[0] + recon.excluded_rows[0]
answers += ["## Reconciliation\n", md]

#### Q1 — top 10 zones by S(z)

In [0]:
_, md = run_query(f"""
SELECT LocationID, zone, borough, ROUND(S, 4) AS S
FROM {STRESS}
ORDER BY ROUND(S, 4) DESC, LocationID ASC
LIMIT 10""")
answers += ["## Q1. Top 10 zones by S(z)\n", md]

#### Q2 — windows on operational day 2025-03-08 and nonexistent timestamps

In [0]:
q2, md_count = run_query(f"""
SELECT COUNT(*) AS windows FROM {GRID}
WHERE operational_date = DATE'2025-03-08' AND window_exists""")
_, md_dropped = run_query(f"""
SELECT window_idx, window_start FROM {GRID}
WHERE operational_date = DATE'2025-03-08' AND NOT window_exists ORDER BY window_idx""")
_, md_dst = run_query(f"""
SELECT SUM(CASE WHEN from_utc_timestamp(to_utc_timestamp(tpep_pickup_datetime, '{TIMEZONE}'), '{TIMEZONE}')
                     != tpep_pickup_datetime THEN 1 ELSE 0 END) AS pickups_at_nonexistent_time,
       SUM(CASE WHEN from_utc_timestamp(to_utc_timestamp(tpep_dropoff_datetime, '{TIMEZONE}'), '{TIMEZONE}')
                     != tpep_dropoff_datetime THEN 1 ELSE 0 END) AS dropoffs_at_nonexistent_time
FROM {BRONZE}""")
answers += ["## Q2. Windows in operational day 2025-03-08\n",
            f"**{int(q2.windows[0])} windows.** The day runs 2025-03-08 04:30 → 2025-03-09 04:30 on the wall clock "
            "(72 windows of 20 min). DST starts at 2025-03-09 02:00 and clocks jump to 03:00, so 02:00–02:59 never "
            "existed. Windows with no real instant are dropped; 01:50–02:10 and 02:50–03:10 contain real time and "
            "stay.\n", md_count, "Dropped windows:\n", md_dropped,
            "**Nonexistent timestamps in the source** are detected with a wall → UTC → wall round trip and excluded "
            "by rule `NONEXISTENT_LOCAL_TIME` (not shifted). A time that cannot exist means the device clock was "
            "wrong, so neither its window nor its duration can be trusted.\n", md_dst]

#### Q3 — duration_min vs naive dropoff_ts − pickup_ts

In [0]:
q3, md = run_query(f"""
SELECT ROUND((CAST(dropoff_ts AS DOUBLE) - CAST(pickup_ts AS DOUBLE)) / 60 - duration_min, 4) AS naive_minus_true_min,
       COUNT(*) AS trips
FROM {FACT}
WHERE ABS((CAST(dropoff_ts AS DOUBLE) - CAST(pickup_ts AS DOUBLE)) / 60 - duration_min) > 1e-9
GROUP BY 1 ORDER BY 1""")
_, md_examples = run_query(f"""
SELECT pickup_ts, dropoff_ts, (CAST(dropoff_ts AS DOUBLE) - CAST(pickup_ts AS DOUBLE)) / 60 AS naive_min, duration_min
FROM {FACT}
WHERE ABS((CAST(dropoff_ts AS DOUBLE) - CAST(pickup_ts AS DOUBLE)) / 60 - duration_min) > 1e-9
ORDER BY pickup_ts, trip_key LIMIT 5""")
answers += ["## Q3. duration_min vs naive dropoff_ts − pickup_ts\n",
            f"**{int(q3.trips.sum()) if len(q3) else 0:,} kept trips differ.** These trips started before 02:00 EST "
            "and ended after 03:00 EDT on 2025-03-09. The wall clock skipped an hour, so naive subtraction "
            "overstates them by 60 minutes. `duration_min` converts both ends to UTC first.\n",
            md, "Examples:\n", md_examples]

#### Q4 — conventional settings: zones entering / leaving the top 10

In [0]:
_, md = run_query(f"""
WITH spec AS (SELECT LocationID, zone, ROUND(S, 4) AS S,
                     ROW_NUMBER() OVER (ORDER BY ROUND(S, 4) DESC, LocationID) AS rnk FROM {STRESS}),
     conv AS (SELECT LocationID, ROUND(S, 4) AS S,
                     ROW_NUMBER() OVER (ORDER BY ROUND(S, 4) DESC, LocationID) AS rnk FROM {STRESS_CONV})
SELECT s.LocationID, s.zone,
       CASE WHEN s.rnk <= 10 THEN 'left top 10' ELSE 'entered top 10' END AS change,
       s.rnk AS rank_spec, s.S AS S_spec, c.rnk AS rank_conventional, c.S AS S_conventional
FROM spec s JOIN conv c ON s.LocationID = c.LocationID
WHERE (s.rnk <= 10) <> (c.rnk <= 10)
ORDER BY change, s.rnk""")
answers += ["## Q4. Conventional settings (day 00:00, windows :00/:20/:40, morning 06:40–09:40)\n",
            "Same notebooks run with widget `settings = conventional` → tables `*_conventional`.\n", md,
            " **Explanation (<100 words):** _TODO after the run._\n"]

#### Q5 — most surprising finding (write by hand from `outputs/profile_report.md`)

In [0]:
answers += ["## Q5. Most surprising thing in the raw data\n",
            "✍️ _TODO (<150 words): pick one finding from `outputs/profile_report.md`; paste its query and output._\n"]

with open(f"{OUTPUT_DIR}/ANSWERS.md", "w") as f:
    f.write("\n".join(answers))
print(f"Saved {OUTPUT_DIR}/ANSWERS.md")